In [1]:
%pip install numpy pdfplumber torch faiss-cpu transformers sentence-transformers tqdm flash-attention

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 1.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207

This notebook is for testing Retrieval-Augmented Generation (RAG) on SEC 10-K data.

In [2]:
import os
import json
import faiss
import torch
import pdfplumber
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from pathlib import Path

In [3]:
# Confirm we are using a suitable runtime
!nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits

import torch

def check_gpu_memory():
    if torch.cuda.is_available():
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU Memory: {gpu_memory:.2f} GB")
        if gpu_memory >= 40:
            print("Sufficient VRAM: At least 40 GB available")
        else:
            print("Insufficient VRAM: Less than 40 GB available")
    else:
        print("No GPU available")

check_gpu_memory()

40960
GPU Memory: 42.47 GB
Sufficient VRAM: At least 40 GB available


In [4]:
# Load the model, tokenizer, and dataset
from google.colab import drive
drive.mount('/content/drive')

import os

# Set up the cache directory
cache_dir = "/content/drive/My Drive/huggingface_cache"
os.makedirs(cache_dir, exist_ok=True)

# Model and device setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
retriever_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2').to(device)
rags_model = AutoModelForCausalLM.from_pretrained('microsoft/Phi-3.5-mini-instruct',
                                                 cache_dir=cache_dir,
                                                 trust_remote_code=True).to(device)
tokenizer = AutoTokenizer.from_pretrained('microsoft/Phi-3.5-mini-instruct',
                                                 cache_dir=cache_dir,
                                                 trust_remote_code=True)
generator = pipeline('text-generation', model=rags_model, tokenizer=tokenizer, device=0 if device=='cuda' else -1)

Mounted at /content/drive


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-mini-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


In [5]:
# FAISS index setup
index_path = 'faiss_index.bin'
corpus_path = 'corpus.json'

# Global variables
index = None
corpus = []

# Load FAISS index and corpus if they exist
if os.path.exists(index_path):
    index = faiss.read_index(index_path)
else:
    index = faiss.IndexFlatL2(384)  # Initialize an empty FAISS index

if os.path.exists(corpus_path):
    with open(corpus_path, 'r') as f:
        corpus = json.load(f)

In [32]:
# Load and preprocess PDFs
def extract_text_from_pdf(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        text = '\n'.join(page.extract_text() or '' for page in pdf.pages)
    return text

# Chunking function
def chunk_text(text, chunk_size=500):
    words = text.split()
    return [' '.join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

# Build FAISS index
def build_index(data_folder):
    global index, corpus
    pdf_files = list(Path(data_folder).rglob('*.pdf'))
    for pdf_file in tqdm(pdf_files, desc='Processing PDFs'):
        text = extract_text_from_pdf(pdf_file)
        chunks = chunk_text(text)
        corpus.extend(chunks)
        embeddings = retriever_model.encode(chunks, convert_to_numpy=True)
        index.add(embeddings)
    faiss.write_index(index, 'faiss_index.bin')
    with open('corpus.json', 'w') as f:
        json.dump(corpus, f)

# Retrieval function
def retrieve_relevant_chunks(query, top_k=5):
    query_embedding = retriever_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)

    if not corpus or any(i >= len(corpus) for i in indices[0]):
        print("Warning: Corpus is empty or indices are out of range. Returning empty list.")
        return []
    return [corpus[i] for i in indices[0]]

# RAG generation function
def generate_response(query):
    # query = query.replace("EGNIVIA", "NVIDIA") # Replace to pull the correct chunks
    relevant_chunks = retrieve_relevant_chunks(query)
    # query = query.replace("NVIDIA", "EGNIVIA") # Replace prior to inference
    context = '\n'.join(relevant_chunks)
    # context = context.replace("NVIDIA", "EGNIVIA") # Replace context to avoid inteference from existing training data
    input_prompt = tokenizer.apply_chat_template([{"role": "system", "content": f"You are a financial analyst."}, {"role": "user", "content": f"Context:\n{context}\n\nQuery:\n{query}"}], tokenize=False, add_generation_prompt=True, return_tensors="pt")
    response = generator(input_prompt, max_new_tokens=256, do_sample=True)
    return response[0]['generated_text']


In [16]:
build_index('/content/drive/My Drive/datasets/temp')

Processing PDFs: 100%|██████████| 1/1 [00:19<00:00, 19.29s/it]


In [8]:
#if __name__ == '__main__':
    # import argparse
    # parser = argparse.ArgumentParser()
    # parser.add_argument('--data_folder', type=str, required=True)
    # parser.add_argument('--query', type=str, required=False, default=None)
    # args = parser.parse_args()

    # if args.query:
    #     print(generate_response(args.query))
    # else:
    #     build_index(args.data_folder)


In [33]:
generate_response("What is NVIDIA's 2023 revenue?")

"<|system|>\nYou are a financial analyst.<|end|>\n<|user|>\nContext:\nMarch 12, 2013 62 Table of Contents NVIDIA CORPORATION AND SUBSIDIARIES CONSOLIDATED STATEMENTS OF INCOME (In thousands, except per share data) Year Ended January 27, January 29, January 30, 2013 2012 2011 Revenue $ 4,280,159 $ 3,997,930 $ 3,543,309 Cost of revenue 2,053,816 1,941,413 2,134,219 Gross profit 2,226,343 2,056,517 1,409,090 Operating expenses: Research and development 1,147,282 1,002,605 848,830 Sales, general and administrative 430,822 405,613 361,513 Legal settlement — — (57,000) Total operating expenses 1,578,104 1,408,218 1,153,343 Income from operations 648,239 648,299 255,747 Interest income 19,908 19,149 19,057 Interest expense (3,294) (3,089) (3,127) Other expense, net (2,814) (963) (508) Income before income tax 662,039 663,396 271,169 Income tax expense 99,503 82,306 18,023 Net income $ 562,536 $ 581,090 $ 253,146 Basic net income per share $ 0.91 $ 0.96 $ 0.44 Weighted average shares used in b

In [27]:
generate_response("What is NVIDIA's 2023 target market?")

"<|system|>\nYou are a financial analyst.<|end|>\n<|user|>\nContext:\nand AI. Our two reportable segments - GPU and Tegra Processor - are based on a single underlying graphics architecture. From our proprietary processors, we have created platforms that address four large markets where our expertise is critical: Gaming, Professional Visualization, Data Center, and Automotive. Our GPU product brands are aimed at specialized markets including GeForce for gamers; Quadro for designers; Tesla and DGX for AI data scientists and big data researchers; and GRID for cloud-based visual computing users. Our Tegra brand incorporates GPUs and multi-core CPUs to drive supercomputing for autonomous robots, drones, and cars, as well as for game consoles and mobile gaming and entertainment devices. Headquartered in Santa Clara, California, NVIDIA was incorporated in California in April 1993 and reincorporated in Delaware in April 1998. Recent Developments, Future Objectives and Challenges Fiscal Year 20

In [15]:
generate_response("What are the primary differences between EGNIVIA's 2022 and 2023 SEC 10-K filings?")

'<|system|>\n9/16/2016 4.5 Form of 2026 Note 8-K 0-23985 Annex B-1 to 9/16/2016 Exhibit 4.2 4.6* Description of Securities 4.7 Officers’ Certificate, dated as of March 31, 2020 8-K 0-23985 4.2 3/31/2020 4.8 Form of 2030 Note 8-K 0-23985 Annex A-1 to 3/31/2020 Exhibit 4.2 4.9 Form of 2040 Note 8-K 0-23985 Annex B-1 to 3/31/2020 Exhibit 4.2 4.10 Form of 2050 Note 8-K 0-23985 Annex C-1 to 3/31/2020 Exhibit 4.2 4.11 Form of 2060 Note 8-K 0-23985 Annex D-1 to 3/31/2020 Exhibit 4.2 4.12 Officers\' Certificate, dated as of June 16, 2021 8-K 0-23985 4.2 6/16/2021 4.13 Form of 2023 Note 8-K 0-23985 Annex A-1 to 6/16/2021 Exhibit 4.2 4.14 Form of 2024 Note 8-K 0-23985 Annex B-1 to 6/16/2021 Exhibit 4.2 4.15 Form of 2028 Note 8-K 0-23985 Annex C-1 to 6/16/2021 Exhibit 4.2 4.16 Form of 2031 Note 8-K 0-23985 Annex D-1 to 6/16/2021 Exhibit 4.2 10.1 Form of Indemnity Agreement between NVIDIA Corporation 8-K 0-23985 10.1 3/7/2006 and each of its directors and officers 10.2+* Amended and Restated 2007 